In [1]:
# Full pipeline: raw config extraction -> calibration standardization -> mapping
#
# This notebook demonstrates the complete workflow:
# 1. Read raw file configurations and save them
# 2. Read manufacturer calibration files and save to single-channel standardized files
# 3. Load single-channel files and match raw file channels to calibration data
# 4. Show how to use the resulting mapping files
#
# The single-channel calibration files produced in step 2 are the canonical
# intermediate format. The same files can also be provided directly by a user
# (see user_provided_cal_pipeline.ipynb) without running steps 1-2.

from pathlib import Path

# Raw file reading
from aa_si_calibration.raw_reader_api import process_raw_folder, save_yaml

# Calibration file parsing and standardized format
from aa_si_calibration import manufacturer_file_parsers
from aa_si_calibration import standardized_file_lib

# Mapping algorithm and pipeline functions
from aa_si_calibration.mapping_algorithm import (
    load_raw_configs,
    load_calibration_data_from_single_files,
    build_mapping,
    get_calibration,
    get_calibration_from_file,
    save_mapping_files,
    print_mapping_preview,
    handle_unused_calibration_files,
    resolve_conflicts_interactive,
    check_for_conflicts,
    check_required_calibration_params,
    verify_calibration_file_usage,
)

from aa_si_calibration.standardized_file_lib import (
    remap_to_short_keys,
    print_short_key_summary,
)

print("All imports successful!")

All imports successful!


In [2]:
# Configuration - define input and output paths

# If True, unused/rejected calibration files are moved to an "unused" folder
# instead of being permanently deleted.
KEEP_UNUSED_STANDARDIZED_FILES = True

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# Cruise identifier for the calibration records
CRUISE_ID = "Pipeline_Example"

# Filename style for single-channel calibration files:
#   False -> long filenames derived from the full calibration key
#   True  -> short filenames: <date>_<freq_hz>_config-<N>.yml
SHORT_FILENAMES = True

# Conflict resolution strategy when a raw channel matches multiple cal files:
#   "interactive" -> prompt the user to choose which file to keep
#   "error"       -> raise an error listing the conflicts; the user must
#                     manually delete the unwanted file(s) and re-run the cell
CONFLICT_RESOLUTION = "error"

# Input folders
RAW_INPUT_FOLDER = Path("./example_data/HB2407_raw")
# RAW_INPUT_FOLDER = Path("./example_data/ek80_CW_raw_file_input_folder")

CAL_INPUT_FOLDER = Path("./example_data/HB2407_cal")
# CAL_INPUT_FOLDER = Path("./example_data/ek80_cal_file_input_folder")

# Output folder with subfolders for organization
OUTPUT_BASE = Path("./HB2407_Outputs")

# Create output subdirectories
RAW_CONFIGS_OUTPUT = OUTPUT_BASE / "raw_file_configs"
SINGLE_CAL_OUTPUT = OUTPUT_BASE / "single_channel_calibration_files"
MAPPING_OUTPUT = OUTPUT_BASE / "mapping_files"
UNUSED_CAL_OUTPUT = OUTPUT_BASE / "unused_calibration_files"
LOGS_OUTPUT = OUTPUT_BASE / "logs"

# Create all output directories
for folder in [RAW_CONFIGS_OUTPUT, SINGLE_CAL_OUTPUT, MAPPING_OUTPUT, LOGS_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Create unused folder only when keeping files
if KEEP_UNUSED_STANDARDIZED_FILES:
    UNUSED_CAL_OUTPUT.mkdir(parents=True, exist_ok=True)

# Output file paths
RAW_CONFIGS_PATH = RAW_CONFIGS_OUTPUT / "raw_file_configs.yaml"
MAPPING_PATH = MAPPING_OUTPUT / "channel_to_calibration_mapping.yaml"
CALIBRATION_DICT_PATH = MAPPING_OUTPUT / "calibration_configurations.yaml"

# Global calibration parameters applied to all single-channel files
global_params = {
    "cruise_id": CRUISE_ID,
    "record_author": RECORD_AUTHOR,
}

print(f"Record author: {RECORD_AUTHOR}")
print(f"Cruise ID: {CRUISE_ID}")
print(f"Short filenames: {SHORT_FILENAMES}")
print(f"Keep unused standardized files: {KEEP_UNUSED_STANDARDIZED_FILES}")
print(f"Conflict resolution mode: {CONFLICT_RESOLUTION}")
print(f"\nInput folders:")
print(f"  Raw files:        {RAW_INPUT_FOLDER}")
print(f"  Calibration files: {CAL_INPUT_FOLDER}")
print(f"\nOutput folders:")
print(f"  Raw configs:              {RAW_CONFIGS_OUTPUT}")
print(f"  Single-channel cal files: {SINGLE_CAL_OUTPUT}")
print(f"  Mapping files:            {MAPPING_OUTPUT}")
print(f"  Logs:                     {LOGS_OUTPUT}")
if KEEP_UNUSED_STANDARDIZED_FILES:
    print(f"  Unused cal files:         {UNUSED_CAL_OUTPUT}")

Record author: Placeholder Author
Cruise ID: Pipeline_Example
Short filenames: True
Keep unused standardized files: True
Conflict resolution mode: error

Input folders:
  Raw files:        example_data\HB2407_raw
  Calibration files: example_data\HB2407_cal

Output folders:
  Raw configs:              HB2407_Outputs\raw_file_configs
  Single-channel cal files: HB2407_Outputs\single_channel_calibration_files
  Mapping files:            HB2407_Outputs\mapping_files
  Logs:                     HB2407_Outputs\logs
  Unused cal files:         HB2407_Outputs\unused_calibration_files


In [3]:
# Step 1: Read raw file configurations
# Process all .raw files in the input folder and extract channel configurations.
# Results are sorted by metadata_start_time (earliest first).

file_configs, frequencies_set = process_raw_folder(RAW_INPUT_FOLDER, verbose=True)

# Save raw file configurations to YAML
save_yaml(file_configs, RAW_CONFIGS_PATH)
print(f"\nSaved raw file configurations to: {RAW_CONFIGS_PATH}")

Found 5 raw files in example_data\HB2407_raw
  - D20241112-T035837.raw
  - D20241112-T052842.raw
  - D20241112-T055441.raw
  - D20241112-T061349.raw
  - D20241112-T064333.raw
File: D20241112-T035837.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 85195
  Valid GPS fixes: 3277
  First GPS: 41.520534, -71.344097
File: D20241112-T052842.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 40036
  Valid GPS fixes: 1540
  First GPS: 41.520675, -71.344127
File: D20241112-T055441.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 29317
  Valid GPS fixes: 1128
  First GPS: 41.520653, -71.344102
File: D20241112-T061349.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 45745
  Valid GPS fixes: 1760
  First GPS: 41.520657, -71.344110
File: D20241112-T064333.raw
Instr

In [4]:
# Step 2: Read manufacturer calibration files and save as single-channel files
# Automatically detects EK60 (.cal) or EK80 (.xml) calibration files,
# parses them, validates against the standardized schema, and saves each
# channel as an individual single-channel .yml file.
#
# These single-channel files are the canonical intermediate format shared
# by both this pipeline (manufacturer files) and the user-provided pipeline.

# Auto-detect calibration file type and extract parameters
# (frequencies from raw files are passed for EK60 sorting; the library sorts them)
cal_params, env_params, other_params, cal_file_type = \
    manufacturer_file_parsers.extract_and_convert_calibration_params(
        CAL_INPUT_FOLDER,
        nc_frequencies=frequencies_set,
        output_logs_folder=LOGS_OUTPUT
    )

print(f"\nParsed {cal_file_type} calibration parameters summary:")
print(f"Channels: {other_params.get('channel')}")
print(f"Frequencies: {other_params.get('frequency_nominal')}")
print(f"Gain corrections: {cal_params.get('gain_correction')}")
print(f"Sa corrections: {cal_params.get('sa_correction')}")
print(f"Equivalent beam angles: {cal_params.get('equivalent_beam_angle')}")

Detected calibration file type: EK80
Found 5 EK80 XML calibration files in example_data\HB2407_cal

Parsing: CalibrationDataFile-D20241112-T040149-018kHz.xml
   Extracted parameters for ES18_18-18kHz
      Frequency range: 18000 - 18000 Hz (1 points)
      Gain range: 23.14 - 23.14 dB
      Power: 1000 W
      Pulse Form: CW

Parsing: CalibrationDataFile-D20241112-T040149-038kHz.xml
   Extracted parameters for ES38-7_38-38kHz
      Frequency range: 38000 - 38000 Hz (1 points)
      Gain range: 26.82 - 26.82 dB
      Power: 1000 W
      Pulse Form: CW

Parsing: CalibrationDataFile-D20241112-T040149-070kHz.xml
   Extracted parameters for ES70-7C_70-70kHz
      Frequency range: 70000 - 70000 Hz (1 points)
      Gain range: 27.54 - 27.54 dB
      Power: 750 W
      Pulse Form: CW

Parsing: CalibrationDataFile-D20241112-T040149-120kHz.xml
   Extracted parameters for ES120-7C_120-120kHz
      Frequency range: 120000 - 120000 Hz (1 points)
      Gain range: 27.05 - 27.05 dB
      Power: 250 W

In [5]:
# Step 2 (continued): Save calibration data as single-channel files
# Each channel is saved as an individual .yml file. These files serve as the
# canonical intermediate format - the same format a user can provide directly
# in the user-provided calibration pipeline.

saved_count, _, standardized_dict = standardized_file_lib.save_single_channel_files_from_params(
    cal_params, 
    env_params, 
    other_params, 
    global_params, 
    output_dir=SINGLE_CAL_OUTPUT,
    short_filenames=SHORT_FILENAMES,
)

print(f"\nSaved {saved_count} single-channel calibration file(s) to: {SINGLE_CAL_OUTPUT}")

print("\nSingle-channel calibration files:")
for f in sorted(SINGLE_CAL_OUTPUT.glob("*.yaml")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

data validated by json schema
data validated by json schema
data validated by json schema
data validated by json schema
data validated by json schema

Saved 5 single-channel calibration file(s) to: HB2407_Outputs\single_channel_calibration_files

Single-channel calibration files:
  2024-11-12__120000__config-1.yaml (1.6 KB)
  2024-11-12__18000__config-1.yaml (1.6 KB)
  2024-11-12__200000__config-1.yaml (1.6 KB)
  2024-11-12__38000__config-1.yaml (1.6 KB)
  2024-11-12__70000__config-1.yaml (1.6 KB)


In [6]:
# Step 3: Load raw file configurations
# Load the raw file configurations we saved in Step 1.
# Calibration data is loaded in the next cell (together with the mapping step)
# so that deleting calibration files and re-running that cell is sufficient.

raw_file_configs = load_raw_configs(RAW_CONFIGS_PATH)

print(f"Loaded {len(raw_file_configs)} raw file configurations")
print(f"\nRaw files: {[f['filename'] for f in raw_file_configs]}")

Loaded 5 raw file configurations

Raw files: ['D20241112-T035837.raw', 'D20241112-T052842.raw', 'D20241112-T055441.raw', 'D20241112-T061349.raw', 'D20241112-T064333.raw']


In [7]:
# Step 3 (continued): Load calibration data and build mapping
# Loads the single-channel calibration files, then matches raw channels to
# calibration data. If any raw channel matches multiple calibration files,
# the conflict is resolved according to CONFLICT_RESOLUTION:
#   "interactive" -> you will be prompted to choose which file to keep
#   "error"       -> a ValueError is raised listing the conflicts; delete
#                     the unwanted file(s) from SINGLE_CAL_OUTPUT and re-run
#                     this cell

# Load calibration data from the single-channel files directory
calibration_data = load_calibration_data_from_single_files(SINGLE_CAL_OUTPUT)

print(f"Loaded {len(calibration_data['channels'])} calibration channel(s) "
      f"from {SINGLE_CAL_OUTPUT}")

# Build the mapping
result = build_mapping(raw_file_configs, calibration_data, verbose=False)
result.print_summary()

# Handle unused calibration files (move or delete)
handle_unused_calibration_files(
    result, calibration_data, SINGLE_CAL_OUTPUT,
    keep_unused=KEEP_UNUSED_STANDARDIZED_FILES,
    unused_dir=UNUSED_CAL_OUTPUT,
)

# Resolve any multiple-match conflicts
if CONFLICT_RESOLUTION == "interactive":
    resolve_conflicts_interactive(
        result, SINGLE_CAL_OUTPUT,
        keep_unused=KEEP_UNUSED_STANDARDIZED_FILES,
        unused_dir=UNUSED_CAL_OUTPUT,
    )
elif CONFLICT_RESOLUTION == "error":
    check_for_conflicts(result, cal_files_dir=SINGLE_CAL_OUTPUT)
else:
    raise ValueError(
        f"Unknown CONFLICT_RESOLUTION mode: {CONFLICT_RESOLUTION!r}. "
        f"Use 'interactive' or 'error'."
    )

# Access the dictionaries from the result
mapping_dict = result.mapping_dict
calibration_dict = result.calibration_dict

Loaded 5 calibration channel(s) from HB2407_Outputs\single_channel_calibration_files

Matching summary:
  Raw file channels:
    Total processed: 25
    Unique: 5
    Matched: 25
    Matched unique: 5
    Unmatched: 0
    Multiplexing warnings: 0
  Calibration files:
    Loaded: 5
    Used: 5
    Conflicts: 0


In [8]:
# Step 3 (continued): Save mapping files and preview results

# Preview the mapping results
print_mapping_preview(result)

# Save the mapping and calibration dictionaries
mapping_path, calibration_path = save_mapping_files(
    result, MAPPING_OUTPUT, short_filenames=SHORT_FILENAMES
)

print(f"\nSaved mapping dictionary to: {mapping_path}")
print(f"Saved calibration dictionary to: {calibration_path}")
print(f"\nNote: Single-channel calibration files already exist in: {SINGLE_CAL_OUTPUT}")

# When using short filenames, update in-memory dicts to match the saved files
if SHORT_FILENAMES:
    mapping_dict, calibration_dict, short_map = remap_to_short_keys(
        mapping_dict, calibration_dict
    )
    print_short_key_summary(short_map, result.calibration_dict)

Mapping dictionary:

D20241112-T035837.raw:
  WBT 400479-15 ES18_ES
    -> 2024-11-12__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2024-11-12__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2024-11-12__70000__config-1
  WBT 400517-15 ES120-7C_ES
    -> 2024-11-12__120000__config-1
  WBT 400509-15 ES200-7C_ES
    -> 2024-11-12__200000__config-1

D20241112-T052842.raw:
  WBT 400479-15 ES18_ES
    -> 2024-11-12__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2024-11-12__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2024-11-12__70000__config-1
  WBT 400517-15 ES120-7C_ES
    -> 2024-11-12__120000__config-1
  WBT 400509-15 ES200-7C_ES
    -> 2024-11-12__200000__config-1

D20241112-T055441.raw:
  WBT 400479-15 ES18_ES
    -> 2024-11-12__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2024-11-12__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2024-11-12__70000__config-1
  WBT 400517-15 ES120-7C_ES
    -> 2024-11-12__120000__config-1
  WBT 400509-15 ES200-7C_ES
    -> 2024-11

In [9]:
# Check for missing required calibration parameters
# For each calibration entry used in the mapping, check that all required
# calibration parameters are present (non-null). Mapping/configuration
# parameters (transceiver_id, transducer_model, etc.) are excluded - they
# were already validated during the matching step.

missing = check_required_calibration_params(calibration_dict)

Checking required calibration parameters...
  All required calibration parameters are present.


In [10]:
# Verify: all remaining calibration files are used
# Confirm that every single-channel calibration file in SINGLE_CAL_OUTPUT is
# referenced by the mapping.

unused_files = verify_calibration_file_usage(calibration_dict, SINGLE_CAL_OUTPUT)

Checking calibration file usage...
  All calibration files are used in the mapping.


In [11]:
# Step 4: How to use the mapping files

print("How to use the mapping files")
print("-" * 40)
print("""
The pipeline produces several output files that can be used independently:

1. Raw file configurations (raw_file_configs.yaml):
   - Contains channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. Single-channel calibration files (single_channel_calibration_files/):
   - One .yml file per channel with calibration parameters
   - This is the canonical intermediate format
   - Can be provided directly by a user (see user_provided_cal_pipeline.ipynb)

3. Mapping dictionary (channel_to_calibration_mapping.yaml):
   - Maps each raw file + channel to a calibration configuration key
   - Allows quick lookup of which calibration to use

4. Calibration configurations (calibration_configurations.yaml):
   - Contains the actual calibration parameters indexed by key
   - Used together with the mapping dictionary
""")

How to use the mapping files
----------------------------------------

The pipeline produces several output files that can be used independently:

1. Raw file configurations (raw_file_configs.yaml):
   - Contains channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. Single-channel calibration files (single_channel_calibration_files/):
   - One .yml file per channel with calibration parameters
   - This is the canonical intermediate format
   - Can be provided directly by a user (see user_provided_cal_pipeline.ipynb)

3. Mapping dictionary (channel_to_calibration_mapping.yaml):
   - Maps each raw file + channel to a calibration configuration key
   - Allows quick lookup of which calibration to use

4. Calibration configurations (calibration_configurations.yaml):
   - Contains the actual calibration parameters indexed by key
   - Used together with the mapping dictionary



In [12]:
# Example: Loading and using the mapping files

# First, let's see what raw files and channels are available
print("Available raw files and channels:")
for filename, channels in mapping_dict.items():
    print(f"\n  {filename}:")
    for channel_id, cal_key in channels.items():
        print(f"    - {channel_id}")
        print(f"      -> maps to: {cal_key}")

# Now demonstrate retrieving calibration for a specific file/channel
print("\nRetrieving calibration data")
print("-" * 40)

# Get the first file and first channel as an example
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Method 1: Use the in-memory dictionaries
        cal_data = get_calibration(example_filename, example_channel, mapping_dict, calibration_dict)
        
        if cal_data:
            print(f"\nCalibration for '{example_filename}' -> '{example_channel}':")
            print(f"  Calibration date: {cal_data.get('calibration_date')}")
            print(f"  Frequency: {cal_data.get('frequency')}")
            print(f"  Gain correction: {cal_data.get('gain_correction')}")
            print(f"  Sa correction: {cal_data.get('sa_correction')}")
            print(f"  Equivalent beam angle: {cal_data.get('equivalent_beam_angle')}")
            print(f"  Sound speed: {cal_data.get('sound_speed_indicative')}")
            print(f"  Absorption: {cal_data.get('absorption_indicative')}")
        else:
            print(f"No calibration found for {example_filename} -> {example_channel}")

Available raw files and channels:

  D20241112-T035837.raw:
    - WBT 400479-15 ES18_ES
      -> maps to: 2024-11-12__18000__config-1
    - WBT 400528-15 ES38-7_ES
      -> maps to: 2024-11-12__38000__config-1
    - WBT 400503-15 ES70-7C_ES
      -> maps to: 2024-11-12__70000__config-1
    - WBT 400517-15 ES120-7C_ES
      -> maps to: 2024-11-12__120000__config-1
    - WBT 400509-15 ES200-7C_ES
      -> maps to: 2024-11-12__200000__config-1

  D20241112-T052842.raw:
    - WBT 400479-15 ES18_ES
      -> maps to: 2024-11-12__18000__config-1
    - WBT 400528-15 ES38-7_ES
      -> maps to: 2024-11-12__38000__config-1
    - WBT 400503-15 ES70-7C_ES
      -> maps to: 2024-11-12__70000__config-1
    - WBT 400517-15 ES120-7C_ES
      -> maps to: 2024-11-12__120000__config-1
    - WBT 400509-15 ES200-7C_ES
      -> maps to: 2024-11-12__200000__config-1

  D20241112-T055441.raw:
    - WBT 400479-15 ES18_ES
      -> maps to: 2024-11-12__18000__config-1
    - WBT 400528-15 ES38-7_ES
      -> maps 

In [13]:
# Example: Loading from saved files (for use in a separate session)

print("Loading calibration from saved files")
print("-" * 40)
print("""
In a new session, you can load the mapping files and retrieve calibration:

    from aa_si_calibration.mapping_algorithm import get_calibration_from_file
    import yaml
    
    # Load the mapping dictionary
    with open('mapping_files/channel_to_calibration_mapping.yaml', 'r') as f:
        mapping_dict = yaml.safe_load(f)
    
    # Get calibration from individual files
    cal_data = get_calibration_from_file(
        filename='<raw_filename>.raw',
        channel_id='<channel_id>',
        mapping_dict=mapping_dict,
        cal_files_dir='single_channel_calibration_files/'
    )
""")

# Demonstrate loading from individual files
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Load from single-channel calibration files
        cal_from_file = get_calibration_from_file(
            example_filename, 
            example_channel, 
            mapping_dict, 
            SINGLE_CAL_OUTPUT
        )
        
        if cal_from_file:
            print(f"Successfully loaded calibration from file!")
            print(f"  File: {example_filename}")
            print(f"  Channel: {example_channel}")
            print(f"  Gain: {cal_from_file.get('gain_correction')}")
        else:
            print("Could not load calibration from file")

Loading calibration from saved files
----------------------------------------

In a new session, you can load the mapping files and retrieve calibration:

    from aa_si_calibration.mapping_algorithm import get_calibration_from_file
    import yaml

    # Load the mapping dictionary
    with open('mapping_files/channel_to_calibration_mapping.yaml', 'r') as f:
        mapping_dict = yaml.safe_load(f)

    # Get calibration from individual files
    cal_data = get_calibration_from_file(
        filename='<raw_filename>.raw',
        channel_id='<channel_id>',
        mapping_dict=mapping_dict,
        cal_files_dir='single_channel_calibration_files/'
    )

Successfully loaded calibration from file!
  File: D20241112-T035837.raw
  Channel: WBT 400479-15 ES18_ES
  Gain: [23.14]


In [14]:
# Pipeline summary

print("Pipeline complete - output files summary")
print("-" * 40)


def list_files_recursive(folder, indent=0):
    """List all files in a folder recursively."""
    if not folder.exists():
        return
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print("  " * indent + f" {item.name}/")
            list_files_recursive(item, indent + 1)
        else:
            size_kb = item.stat().st_size / 1024
            print("  " * indent + f"  {item.name} ({size_kb:.1f} KB)")


print(f"\nOutput directory: {OUTPUT_BASE}")
print("-" * 40)
list_files_recursive(OUTPUT_BASE)

print("\nNext steps:")
print("""
With these output files, you can:

1. Use the raw_file_configs.yaml to understand your data structure
2. Use the single-channel calibration files with any compatible processing tool
3. Use the mapping files to automatically apply the correct calibration
   to each channel when processing raw data with echopype or similar tools
4. Provide your own single-channel calibration files and run the
   user_provided_cal_pipeline.ipynb to map them to raw files

For echopype calibration, you can extract the relevant parameters:
    - gain_correction
    - sa_correction  
    - equivalent_beam_angle
    - sound_speed_indicative
    - absorption_indicative
""")

Pipeline complete - output files summary
----------------------------------------

Output directory: HB2407_Outputs
----------------------------------------
 logs/
    calibration_flags.json (0.1 KB)
 mapping_files/
    calibration_configurations.yaml (8.9 KB)
    channel_to_calibration_mapping.yaml (1.5 KB)
 raw_file_configs/
    raw_file_configs.yaml (18.2 KB)
 single_channel_calibration_files/
    2024-11-12__120000__config-1.yaml (1.6 KB)
    2024-11-12__18000__config-1.yaml (1.6 KB)
    2024-11-12__200000__config-1.yaml (1.6 KB)
    2024-11-12__38000__config-1.yaml (1.6 KB)
    2024-11-12__70000__config-1.yaml (1.6 KB)
 unused_calibration_files/

Next steps:

With these output files, you can:

1. Use the raw_file_configs.yaml to understand your data structure
2. Use the single-channel calibration files with any compatible processing tool
3. Use the mapping files to automatically apply the correct calibration
   to each channel when processing raw data with echopype or similar tool